# Ground Truth CSV → JSONL Converter

Converts a **flat CSV** ground truth file (all columns present, `NOT_FOUND` for inapplicable fields)
into a **per-type JSONL** file where each record carries only its document type's fields.

Field lists are read from `config/field_definitions.yaml`.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Set these paths before running.  Relative paths are resolved from the
# notebook's parent directory (i.e. the project root).

CSV_INPUT  = "../evaluation_data-Copy1/synthetic_original/ground_truth_synthetic.csv"
JSONL_OUTPUT = "../evaluation_data-Copy1/synthetic/ground_truth.jsonl"

# Normally auto-detected; override only if yours lives elsewhere.
FIELD_DEFS = "../config/field_definitions.yaml"

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml

# ── Resolve paths ──────────────────────────────────────────────────────
csv_path = Path(CSV_INPUT).expanduser().resolve()
jsonl_path = Path(JSONL_OUTPUT).expanduser().resolve()
field_defs_path = Path(FIELD_DEFS).expanduser().resolve()

assert csv_path.exists(), f"CSV not found: {csv_path}"
assert field_defs_path.exists(), f"field_definitions.yaml not found: {field_defs_path}"

print(f"CSV input:   {csv_path}")
print(f"JSONL output: {jsonl_path}")
print(f"Field defs:  {field_defs_path}")

In [ ]:
# ── Load field definitions ────────────────────────────────────────────
with field_defs_path.open() as f:
    defs = yaml.safe_load(f)

type_fields: dict[str, list[str]] = {}
for doc_type, info in defs.get("document_fields", {}).items():
    fields = info.get("fields", [])
    type_fields[doc_type.upper()] = fields

print("Document types and field counts:")
for dt, fields in sorted(type_fields.items()):
    print(f"  {dt:20s} → {len(fields)} fields")

In [ ]:
# ── Read CSV ───────────────────────────────────────────────────────────
df = pd.read_csv(csv_path, dtype=str).fillna("NOT_FOUND")

# First column is always the filename; DOCUMENT_TYPE is the type column.
filename_col = df.columns[0]
assert "DOCUMENT_TYPE" in df.columns, (
    f"No DOCUMENT_TYPE column found. Columns present: {list(df.columns)}"
)
doc_type_col = "DOCUMENT_TYPE"

print(f"Rows: {len(df)}  |  Filename column: '{filename_col}'  |  Type column: '{doc_type_col}'")
print("\nDocument type distribution:")
print(df[doc_type_col].value_counts().to_string())
print("\nCSV preview (first 3 rows):")
df.head(3)

In [ ]:
# ── Convert to JSONL ──────────────────────────────────────────────────
records_written = 0
skipped_types: dict[str, int] = {}
all_records: list[dict[str, str]] = []

for _, row in df.iterrows():
    doc_type = str(row[doc_type_col]).strip().upper()
    filename = str(row[filename_col]).strip()

    # Look up per-type fields
    fields = type_fields.get(doc_type)
    if fields is None:
        skipped_types[doc_type] = skipped_types.get(doc_type, 0) + 1
        continue

    record: dict[str, str] = {"filename": filename}
    for field in fields:
        val = str(row.get(field, "NOT_FOUND")).strip()
        record[field] = val

    all_records.append(record)
    records_written += 1

# Write output
jsonl_path.parent.mkdir(parents=True, exist_ok=True)
with jsonl_path.open("w") as f:
    for rec in all_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {records_written} records to {jsonl_path}")
if skipped_types:
    for dtype, count in sorted(skipped_types.items()):
        print(f"  SKIPPED {count} records with unknown type: {dtype}")

In [ ]:
# ── Preview output ────────────────────────────────────────────────────
print("Output JSONL preview (first 3 records):\n")
for rec in all_records[:3]:
    print(json.dumps(rec, indent=2, ensure_ascii=False))
    print()